In [1]:
# ====== PARAM ======
level_depth = 0  # <--- cambia qui il livello, se ti serve


In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Visualizza il "best" modello HDBSCAN (ultima riga del CSV di BO)
e mostra il barchart dei topic.

Compatibile con la versione BO che salva:
- CSV: con separatore ';' (o ',')
- Modello: nome tipo
  CPU_BO_KPCA_rbf_nc{ncomp}_g{gamma}__HDBSCAN_mcs{mcs}__{embed_model}_fp{fp}
"""

from pathlib import Path
import os
import sys
import argparse
import pandas as pd
from bertopic import BERTopic
import plotly.io as pio

# ---------- Utils ----------
def read_csv_smart(path: Path) -> pd.DataFrame:
    """
    Prova prima con ';' (Excel ITA-friendly), poi con ','.
    Salta eventuali righe problematiche (on_bad_lines='skip').
    """
    try:
        df = pd.read_csv(path, sep=';', engine='python', on_bad_lines='skip')
        if df.shape[1] > 1:
            return df
    except Exception:
        pass
    # fallback a virgola
    return pd.read_csv(path, sep=',', engine='python', on_bad_lines='skip')

# ---------- CLI ----------
parser = argparse.ArgumentParser(
    description="Mostra barchart del modello BERTopic 'best' (HDBSCAN)"
)
parser.add_argument("--level", type=str, default=None,
                    help="level id (es. 0, 1, 2...). Se assente usa $LEVEL_DEPTH o '0'.")
parser.add_argument("--base", type=str,
                    default="/home/students/s328743/Thesis/Smart_crawler_telegram/results/levels",
                    help="Cartella base dei risultati.")
parser.add_argument("--max_topics", type=int, default=180,
                    help="Massimo numero di topic da mostrare nel barchart.")
parser.add_argument("--width", type=int, default=350, help="Larghezza grafico Plotly.")
parser.add_argument("--height", type=int, default=450, help="Altezza grafico Plotly.")

# In Jupyter/IPython arrivano argomenti sconosciuti (-f ...): ignorali.
args, _unknown = parser.parse_known_args()

# -------- LEVEL --------
default_level = "0"
level_depth = args.level or os.environ.get("LEVEL_DEPTH") or default_level
level_depth = str(level_depth)

# Plotly headless-safe
pio.renderers.default = (
    "notebook_connected" if (os.environ.get("DISPLAY") or os.environ.get("MPLBACKEND")) else "png"
)

# ====== PATHS (runtime_grid_search) ======
BASE_LEVEL_DIR = Path(args.base) / f"level_{level_depth}"
RUNTIME_DIR    = BASE_LEVEL_DIR / "runtime_grid_search"
RES_CSV        = RUNTIME_DIR / f"bo_results_level_{level_depth}_KernelPCA_HDBSCAN_CPU.csv"
MODELS_DIR     = RUNTIME_DIR / f"bertopic_models_level_{level_depth}"

print(f"[INFO] level={level_depth}")
print(f"[INFO] RES_CSV   = {RES_CSV}")
print(f"[INFO] MODELS_DIR= {MODELS_DIR}")

# ====== CHECK ======
missing = [p for p in (RES_CSV, MODELS_DIR) if not p.exists()]
if missing:
    miss_str = "\n  - ".join(str(m) for m in missing)
    raise FileNotFoundError(f"Required paths/files not found:\n  - {miss_str}")

# ====== LOAD CSV (auto-sep) ======
df = read_csv_smart(RES_CSV)
if df.empty:
    raise ValueError(f"{RES_CSV} è vuoto o tutte le righe sono state scartate.")
print(f"[INFO] Loaded {len(df)} rows; columns={list(df.columns)}")

# ====== PICK LAST VALID ROW ======
needed = ["embed_model", "kpca_n_components", "kpca_gamma", "hdbscan_min_cluster_size"]
for col in needed:
    if col not in df.columns:
        raise ValueError(f"Manca la colonna '{col}' nel CSV {RES_CSV}")

mask = df[needed].notnull().all(axis=1)
if not mask.any():
    raise ValueError("Nessuna riga valida nel CSV con tutte le colonne richieste non nulle.")

last = df[mask].iloc[-1].copy()

embed_model  = str(last["embed_model"])
ncomp        = int(last["kpca_n_components"])
gamma        = float(last["kpca_gamma"])
mcs          = int(last["hdbscan_min_cluster_size"])

# ====== RICOSTRUISCI NOME MODELLO ======
# Esempio: CPU_BO_KPCA_rbf_nc{ncomp}_g{gamma:.4g}__HDBSCAN_mcs{mcs}__{embed_model}_fp*
prefix_preciso = (
    f"CPU_BO_KPCA_rbf_nc{ncomp}_g{gamma:.4g}__HDBSCAN_mcs{mcs}__{embed_model}_fp"
)
candidates = sorted(MODELS_DIR.glob(prefix_preciso + "*"),
                    key=lambda p: p.stat().st_mtime, reverse=True)

# fallback per arrotondamenti di gamma
if not candidates:
    pattern_fallback = f"CPU_BO_KPCA_rbf_nc{ncomp}_g*__HDBSCAN_mcs{mcs}__{embed_model}_fp*"
    candidates = sorted(MODELS_DIR.glob(pattern_fallback),
                        key=lambda p: p.stat().st_mtime, reverse=True)
else:
    pattern_fallback = None

if not candidates:
    all_models = list(MODELS_DIR.glob("CPU_BO_KPCA_rbf_*"))
    available = ""
    if all_models:
        preview = "\n  - " + "\n  - ".join(p.name for p in sorted(all_models)[:10])
        if len(all_models) > 10:
            preview += f"\n  ... and {len(all_models) - 10} more"
        available = f"\nModelli presenti (primi 10):{preview}"
    raise FileNotFoundError(
        "Nessun modello trovato con i pattern:\n"
        f"  - '{prefix_preciso}*'"
        + (f"\n  - '{pattern_fallback}'" if pattern_fallback else "")
        + available
    )

model_path = candidates[0]
print(f"[INFO] Ultima riga CSV -> ncomp={ncomp}, gamma={gamma:.6f}, mcs={mcs}, embed='{embed_model}'")
print(f"[INFO] Carico il modello: {model_path}")

# ====== LOAD & VISUALIZE ======
topic_model = BERTopic.load(str(model_path))

# stima numero topic
try:
    labels = topic_model.topics_
    n_topics = len(set(labels)) - (1 if -1 in labels else 0)
except Exception:
    topics_dict = topic_model.get_topics()
    n_topics = sum(1 for tid in topics_dict if tid != -1)

n_show = max(1, min(n_topics, int(args.max_topics)))
print(f"[INFO] topics trovati: {n_topics} (mostro al massimo {n_show})")

fig = topic_model.visualize_barchart(
    top_n_topics=n_show,
    n_words=10,
    width=int(args.width),
    height=int(args.height),
)

# prova a mostrare, altrimenti salva HTML
try:
    fig.show()
except Exception as e:
    print(f"[WARN] Impossibile mostrare il grafico in questa sessione: {e}")
    print("[INFO] Salvando il grafico come HTML...")
    output_html = RUNTIME_DIR / f"topics_barchart_level_{level_depth}.html"
    fig.write_html(str(output_html))
    print(f"[INFO] Grafico salvato in: {output_html}")


[INFO] level=0
[INFO] RES_CSV   = /home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_0/runtime_grid_search/bo_results_level_0_KernelPCA_HDBSCAN_CPU.csv
[INFO] MODELS_DIR= /home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_0/runtime_grid_search/bertopic_models_level_0
[INFO] Loaded 4 rows; columns=['embed_model', 'kpca_n_components', 'kpca_kernel', 'kpca_gamma', 'hdbscan_min_cluster_size', 'hdbscan_min_samples_ratio', 'coherence', 'diversity', 'silhouette', 'n_topics', 'seconds', 'avg_score', 'penalty', 'final_score', 'coh_norm', 'div_norm', 'sil_norm']
[INFO] Ultima riga CSV -> ncomp=25, gamma=0.280242, mcs=17, embed='all-MiniLM-L6-v2'
[INFO] Carico il modello: /home/students/s328743/Thesis/Smart_crawler_telegram/results/levels/level_0/runtime_grid_search/bertopic_models_level_0/CPU_BO_KPCA_rbf_nc25_g0.2802__HDBSCAN_mcs17__all-MiniLM-L6-v2_fp1
[INFO] topics trovati: 4 (mostro al massimo 4)
